# 02 - Generate Mock Data and Ingest to Bronze

**Purpose:** Sample real MBIDs from Bronze ListenBrainz data, generate HG Media mock master and transactional data anchored to real MBIDs, upload everything to Bronze.

**Output Bronze paths:**
```
bronze/artists/
bronze/tracks/
bronze/contributors/
bronze/channels/
bronze/videos/
bronze/platforms/
bronze/distributors/
bronze/youtube_rate/
bronze/distributor_revenue_monthly/
bronze/youtube_view_daily/
bronze/content_contributor_mapping/
```

## 0. Imports and Configuration

In [8]:
import boto3
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import numpy as np
import os
import io
import random
import uuid
from datetime import datetime, timedelta
from faker import Faker

fake = Faker()
random.seed(42)
np.random.seed(42)

# ── MinIO Connection ─────────────────────────────────────────────────
MINIO_ENDPOINT   = "http://storage:9000"
MINIO_ACCESS_KEY = os.environ.get("MINIO_ROOT_USER", "root")
MINIO_SECRET_KEY = os.environ.get("MINIO_ROOT_PASSWORD", "password")
BRONZE_BUCKET    = "bronze"
INGESTED_DATE    = "2026-06-28"  # match ListenBrainz ingestion date
LB_PREFIX        = f"listenbrainz/ingested_date={INGESTED_DATE}/"

# ── Scale Config ─────────────────────────────────────────────────────
N_TRACKS         = 3000
N_ARTISTS        = 500
N_CONTRIBUTORS   = 50
N_CHANNELS       = 10
N_VIDEOS         = 300
N_COUNTRIES      = 70
N_MONTHS         = 24  # months of revenue data
START_MONTH      = "2025-01"  # revenue period start

# ── Output dir ───────────────────────────────────────────────────────
OUT_DIR = "/datn/ingestion/generated/"
os.makedirs(OUT_DIR, exist_ok=True)

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1"
)
print("Config OK")
print(f"Scale: {N_TRACKS} tracks, {N_ARTISTS} artists, {N_CONTRIBUTORS} contributors")

Config OK
Scale: 3000 tracks, 500 artists, 50 contributors


## Helper: Upload DataFrame to Bronze

In [10]:
def upload_df_to_bronze(df, prefix, filename):
    """Upload a DataFrame as CSV to MinIO Bronze bucket."""
    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)
    csv_bytes = csv_buffer.getvalue().encode("utf-8")
    s3_key = f"{prefix}{filename}"
    s3.put_object(
        Bucket=BRONZE_BUCKET,
        Key=s3_key,
        Body=csv_bytes,
        ContentType="text/csv"
    )
    print(f"Uploaded → bronze/{s3_key} ({len(df):,} rows)")
    return s3_key

## Step 1: Sample Real MBIDs from Bronze

In [11]:
# Sample from first 10 partitions for speed - enough unique MBIDs
SAMPLE_PARTITIONS = 10

recording_pool = {}  # mbid -> (recording_name, artist_name, release_name, release_mbid)
artist_pool    = {}  # artist_mbid -> artist_name

paginator = s3.get_paginator("list_objects_v2")
pages     = paginator.paginate(Bucket=BRONZE_BUCKET, Prefix=LB_PREFIX)

partition_keys = []
for page in pages:
    for obj in page.get("Contents", []):
        partition_keys.append(obj["Key"])

partition_keys = sorted(
    partition_keys,
    key=lambda x: int(os.path.basename(x).replace(".parquet", ""))
)[:SAMPLE_PARTITIONS]

print(f"Sampling from {len(partition_keys)} partitions...")

for key in partition_keys:
    response = s3.get_object(Bucket=BRONZE_BUCKET, Key=key)
    body     = response["Body"].read()
    buf      = pa.BufferReader(pa.py_buffer(body))
    table    = pq.read_table(buf, columns=[
        "recording_mbid", "recording_name",
        "artist_name", "artist_credit_mbids",
        "release_name", "release_mbid"
    ])
    rows = table.to_pydict()
    for i in range(table.num_rows):
        rec_mbid = rows["recording_mbid"][i]
        if rec_mbid and rec_mbid not in recording_pool:
            recording_pool[rec_mbid] = {
                "recording_name" : rows["recording_name"][i],
                "artist_name"    : rows["artist_name"][i],
                "release_name"   : rows["release_name"][i],
                "release_mbid"   : rows["release_mbid"][i],
            }
        mbids = rows["artist_credit_mbids"][i]
        if mbids:
            for mbid in mbids:
                if mbid and mbid not in artist_pool:
                    artist_pool[mbid] = rows["artist_name"][i]
    print(f"  {os.path.basename(key)}: pool={len(recording_pool):,} recordings, {len(artist_pool):,} artists")

print(f"\nTotal unique recordings : {len(recording_pool):,}")
print(f"Total unique artists    : {len(artist_pool):,}")

Sampling from 10 partitions...
  779.parquet: pool=647,037 recordings, 127,878 artists
  780.parquet: pool=1,014,960 recordings, 173,918 artists
  781.parquet: pool=1,308,048 recordings, 206,365 artists
  782.parquet: pool=1,551,350 recordings, 232,908 artists
  783.parquet: pool=1,763,535 recordings, 254,874 artists
  784.parquet: pool=1,954,246 recordings, 274,269 artists
  785.parquet: pool=2,122,913 recordings, 290,717 artists
  786.parquet: pool=2,283,769 recordings, 305,570 artists
  787.parquet: pool=2,428,841 recordings, 319,147 artists
  788.parquet: pool=2,566,069 recordings, 331,597 artists

Total unique recordings : 2,566,069
Total unique artists    : 331,597


In [12]:
# Sample N_TRACKS and N_ARTISTS from pools
sampled_recording_mbids = random.sample(list(recording_pool.keys()), min(N_TRACKS, len(recording_pool)))
sampled_artist_mbids    = random.sample(list(artist_pool.keys()),    min(N_ARTISTS, len(artist_pool)))

print(f"Sampled {len(sampled_recording_mbids)} tracks")
print(f"Sampled {len(sampled_artist_mbids)} artists")

# Preview
for mbid in sampled_recording_mbids[:3]:
    print(f"  Track: {mbid} → {recording_pool[mbid]['recording_name']} by {recording_pool[mbid]['artist_name']}")

Sampled 3000 tracks
Sampled 500 artists
  Track: 4c957787-9c63-45b4-a44c-1718cd5e95fe → Company by Sir Chloe
  Track: 50bc8d34-8860-4cac-baf5-ff5c9d86fee4 → Gryt by Gaupa
  Track: d65f5032-7134-41dd-a3f0-e32ff1e8a336 → Gift of the Magi 2: Return of the Magi by Andrew Jackson Jihad


## Step 2: Generate Master Data

In [13]:
# ── 2a. Platforms ─────────────────────────────────────────────────────
platforms = pd.DataFrame([
    {"platform_id": "PLT_SPF", "platform_name": "Spotify",     "platform_type": "STREAMING"},
    {"platform_id": "PLT_APM", "platform_name": "Apple Music", "platform_type": "STREAMING"},
    {"platform_id": "PLT_TTK", "platform_name": "TikTok",      "platform_type": "STREAMING"},
    {"platform_id": "PLT_SCL", "platform_name": "SoundCloud",  "platform_type": "STREAMING"},
    {"platform_id": "PLT_YTM", "platform_name": "YouTube Music","platform_type": "STREAMING"},
    {"platform_id": "PLT_YTB", "platform_name": "YouTube",     "platform_type": "VIDEO"},
])
upload_df_to_bronze(platforms, "platforms/", "platforms.csv")

# ── 2b. Distributors ──────────────────────────────────────────────────
distributors = pd.DataFrame([
    {"distributor_id": "DST_RTN", "distributor_name": "Routenote", "payment_cycle": "Monthly", "active_status": "ACTIVE"},
    {"distributor_id": "DST_LDR", "distributor_name": "LANDR",     "payment_cycle": "Monthly", "active_status": "ACTIVE"},
    {"distributor_id": "DST_DTO", "distributor_name": "Ditto",     "payment_cycle": "Monthly", "active_status": "ACTIVE"},
    {"distributor_id": "DST_TNC", "distributor_name": "Tunecore",  "payment_cycle": "Monthly", "active_status": "ACTIVE"},
    {"distributor_id": "DST_OPM", "distributor_name": "ONERPM",    "payment_cycle": "Monthly", "active_status": "ACTIVE"},
    {"distributor_id": "DST_WMG", "distributor_name": "WMG",       "payment_cycle": "Monthly", "active_status": "ACTIVE"},
])
upload_df_to_bronze(distributors, "distributors/", "distributors.csv")

# ── 2c. Countries ─────────────────────────────────────────────────────
COUNTRIES = [
    ("VN", "Vietnam"),      ("US", "United States"), ("GB", "United Kingdom"),
    ("JP", "Japan"),        ("KR", "South Korea"),   ("TH", "Thailand"),
    ("ID", "Indonesia"),    ("MY", "Malaysia"),      ("SG", "Singapore"),
    ("PH", "Philippines"),  ("AU", "Australia"),     ("FR", "France"),
    ("DE", "Germany"),      ("BR", "Brazil"),         ("MX", "Mexico"),
    ("IN", "India"),        ("CA", "Canada"),         ("IT", "Italy"),
    ("ES", "Spain"),        ("NL", "Netherlands"),
]

# ── 2d. YouTube Rate Table ────────────────────────────────────────────
yt_rates = {
    "US": 0.004, "GB": 0.0038, "AU": 0.0035, "CA": 0.0035,
    "DE": 0.003, "FR": 0.003,  "NL": 0.003,  "IT": 0.0028,
    "ES": 0.0025,"JP": 0.003,  "KR": 0.0025, "SG": 0.003,
    "MY": 0.001, "TH": 0.001,  "ID": 0.0008, "PH": 0.0008,
    "VN": 0.0005,"IN": 0.0007, "BR": 0.001,  "MX": 0.001,
}
youtube_rate = pd.DataFrame([
    {
        "country": code,
        "country_name": name,
        "rate_per_view_usd": yt_rates[code],
        "effective_from": "2024-01-01",
        "effective_to": "2026-12-31"
    }
    for code, name in COUNTRIES
])
upload_df_to_bronze(youtube_rate, "youtube_rate/", "youtube_rate.csv")

print("\nMaster reference data uploaded.")

Uploaded → bronze/platforms/platforms.csv (6 rows)
Uploaded → bronze/distributors/distributors.csv (6 rows)
Uploaded → bronze/youtube_rate/youtube_rate.csv (20 rows)

Master reference data uploaded.


In [14]:
# ── 2e. Artists ───────────────────────────────────────────────────────
distributor_ids = distributors["distributor_id"].tolist()
label_names     = ["HG Music", "HG Indie", "HG Urban", "HG Pop", "HG EDM"]
project_names   = ["Project Alpha", "Project Beta", "Project Gamma", "Project Delta", "Project Epsilon"]

artists_rows = []
for i, mbid in enumerate(sampled_artist_mbids):
    artist_name = artist_pool[mbid]
    artists_rows.append({
        "artist_id"       : mbid,
        "artist_name"     : artist_name,
        "distributor_id"  : random.choice(distributor_ids),
        "label_name"      : random.choice(label_names),
        "project_name"    : random.choice(project_names),
        "active_status"   : random.choices(["ACTIVE", "INACTIVE"], weights=[90, 10])[0],
        "created_date"    : fake.date_between(start_date="-5y", end_date="-1y").strftime("%Y-%m-%d")
    })

artists_df = pd.DataFrame(artists_rows)
upload_df_to_bronze(artists_df, "artists/", "artists.csv")
print(f"Sample: {artists_df[['artist_id','artist_name','distributor_id']].head(3).to_string()}")

Uploaded → bronze/artists/artists.csv (500 rows)
Sample:                               artist_id    artist_name distributor_id
0  63ecc430-f223-4dbd-9e59-334284f11ad5   SM Jazz Trio        DST_DTO
1  df8e73d9-8a1e-4312-9a4d-09528ab1062c  Nicky Flowers        DST_RTN
2  d5fcca60-4499-432c-9482-e343813835f7  Deine Freunde        DST_WMG


In [15]:
# ── 2f. Tracks ────────────────────────────────────────────────────────
artist_ids = artists_df["artist_id"].tolist()

tracks_rows = []
for i, mbid in enumerate(sampled_recording_mbids):
    rec    = recording_pool[mbid]
    artist = random.choice(artist_ids)
    dist   = artists_df[artists_df["artist_id"] == artist]["distributor_id"].values[0]
    tracks_rows.append({
        "track_id"        : mbid,
        "track_title"     : rec["recording_name"],
        "asset_id"        : f"AST_T{i+1:04d}",
        "isrc"            : f"VN-HGM-{random.randint(10,99)}-{random.randint(10000,99999)}",
        "recording_mbid"  : mbid,
        "release_mbid"    : rec["release_mbid"],
        "release_name"    : rec["release_name"],
        "artist_id"       : artist,
        "distributor_id"  : dist,
        "label_name"      : random.choice(label_names),
        "project_name"    : random.choice(project_names),
        "release_date"    : fake.date_between(start_date="-4y", end_date="-6m").strftime("%Y-%m-%d"),
        "active_status"   : random.choices(["ACTIVE", "INACTIVE"], weights=[85, 15])[0]
    })

tracks_df = pd.DataFrame(tracks_rows)
upload_df_to_bronze(tracks_df, "tracks/", "tracks.csv")
print(f"Sample:\n{tracks_df[['track_id','track_title','artist_id','distributor_id']].head(3).to_string()}")

Uploaded → bronze/tracks/tracks.csv (3,000 rows)
Sample:
                               track_id                             track_title                             artist_id distributor_id
0  4c957787-9c63-45b4-a44c-1718cd5e95fe                                 Company  2f763cf8-0c8e-450f-82ea-e797b7e046b1        DST_RTN
1  50bc8d34-8860-4cac-baf5-ff5c9d86fee4                                    Gryt  44663f86-64db-4ccd-98b1-8cb122b4e8a6        DST_TNC
2  d65f5032-7134-41dd-a3f0-e32ff1e8a336  Gift of the Magi 2: Return of the Magi  4e645980-0041-4aa7-92e2-9fe836575479        DST_WMG


In [16]:
# ── 2g. Contributors ──────────────────────────────────────────────────
vn_names = [
    "Nguyễn Văn An", "Trần Thị Bình", "Lê Văn Công", "Phạm Thị Dung",
    "Hoàng Văn Em", "Đặng Thị Phương", "Bùi Văn Giang", "Đỗ Thị Hà",
    "Ngô Văn Inh", "Vũ Thị Kim", "Đinh Văn Long", "Trương Thị Mai",
    "Lý Văn Nam", "Phan Thị Oanh", "Mai Văn Phúc", "Cao Thị Quỳnh",
    "Lưu Văn Rồng", "Tăng Thị Sen", "Hồ Văn Thắng", "Dương Thị Uyên",
    "Châu Văn Vinh", "Lâm Thị Xuân", "Kiều Văn Yên", "Tô Thị Zung",
    "Vương Văn Anh", "Quách Thị Bảo", "Tống Văn Cường", "Lạc Thị Diệu",
    "Mạc Văn Ân", "Ninh Thị Bích", "Trịnh Văn Đức", "Bùi Thị Hương",
    "Đoàn Văn Khoa", "Lưu Thị Lan", "Phùng Văn Minh", "Tạ Thị Nga",
    "Vương Văn Phong", "Đinh Thị Quỳnh", "Hà Văn Sơn", "Đặng Thị Tâm",
    "Nguyễn Văn Uy", "Trần Thị Vân", "Lê Văn Xuân", "Phạm Thị Yến",
    "Hoàng Văn Zoan", "Bùi Văn Bảo", "Đỗ Thị Châu", "Ngô Văn Dũng",
    "Vũ Thị Em", "Đinh Văn Phú"
]

# Define BEFORE the loop
roles = ["Producer", "Composer", "Arranger", "Artist Manager",
         "Content Creator", "Video Editor", "Sound Engineer",
         "Mixing Engineer", "Mastering Engineer", "Operator"]
teams = ["Production", "Content", "Distribution", "Marketing", "Operations"]

contributors_rows = []
for i in range(N_CONTRIBUTORS):
    contributors_rows.append({
        "contributor_id"  : f"CTB{i+1:03d}",
        "contributor_name": vn_names[i % len(vn_names)],
        "team"            : random.choice(teams),
        "default_role"    : random.choice(roles),
        "active_status"   : random.choices(["ACTIVE", "INACTIVE"], weights=[90, 10])[0]
    })

contributors_df = pd.DataFrame(contributors_rows)
upload_df_to_bronze(contributors_df, "contributors/", "contributors.csv")
print(contributors_df.head(5).to_string())

Uploaded → bronze/contributors/contributors.csv (50 rows)
  contributor_id contributor_name          team     default_role active_status
0         CTB001    Nguyễn Văn An       Content   Artist Manager        ACTIVE
1         CTB002    Trần Thị Bình       Content         Arranger        ACTIVE
2         CTB003      Lê Văn Công  Distribution         Arranger        ACTIVE
3         CTB004    Phạm Thị Dung     Marketing         Arranger        ACTIVE
4         CTB005     Hoàng Văn Em  Distribution  Content Creator        ACTIVE


In [17]:
# ── 2h. YouTube Channels ──────────────────────────────────────────────
channel_names = [
    "HG Music Official", "HG Indie Vibes", "HG Urban Beats",
    "HG Pop World", "HG EDM Factory", "HG Acoustic Sessions",
    "HG Lo-Fi Studio", "HG Chill Mix", "HG Trending", "HG Underground"
]
channels_rows = []
for i in range(N_CHANNELS):
    channels_rows.append({
        "channel_id"    : f"CHN{i+1:03d}",
        "channel_name"  : channel_names[i],
        "owner_team"    : random.choice(teams),
        "created_date"  : fake.date_between(start_date="-6y", end_date="-2y").strftime("%Y-%m-%d"),
        "active_status" : "ACTIVE"
    })
channels_df = pd.DataFrame(channels_rows)
upload_df_to_bronze(channels_df, "youtube_channels/", "youtube_channels.csv")

# ── 2i. YouTube Videos ────────────────────────────────────────────────
content_types = ["MV", "Lyric Video", "Live Performance", "Behind the Scenes", "Short"]
videos_rows   = []
channel_ids   = channels_df["channel_id"].tolist()

for i in range(N_VIDEOS):
    track = tracks_df.iloc[i % len(tracks_df)]
    videos_rows.append({
        "video_id"     : f"VID{i+1:04d}",
        "video_title"  : f"{track['track_title']} ({random.choice(content_types)})",
        "asset_id"     : f"AST_V{i+1:04d}",
        "channel_id"   : random.choice(channel_ids),
        "track_id"     : track["track_id"],
        "project_name" : track["project_name"],
        "content_type" : random.choice(content_types),
        "upload_date"  : fake.date_between(start_date="-3y", end_date="-1m").strftime("%Y-%m-%d"),
        "editor_code"  : f"ED{random.randint(100,999)}"
    })
videos_df = pd.DataFrame(videos_rows)
upload_df_to_bronze(videos_df, "youtube_videos/", "youtube_videos.csv")
print("Channels and videos uploaded.")

Uploaded → bronze/youtube_channels/youtube_channels.csv (10 rows)
Uploaded → bronze/youtube_videos/youtube_videos.csv (300 rows)
Channels and videos uploaded.


## Step 3: Generate Transactional Data

In [18]:
# ── 3a. Distributor Revenue Monthly ───────────────────────────────────
# Uses real MBIDs as track_id and artist_id

# Streaming rate per platform (USD per stream)
platform_rates = {
    "PLT_SPF": 0.003,    # Spotify
    "PLT_APM": 0.005,    # Apple Music
    "PLT_TTK": 0.0005,   # TikTok
    "PLT_SCL": 0.0025,   # SoundCloud
    "PLT_YTM": 0.002,    # YouTube Music
}

# Country weight (higher = more streams from that country)
country_weights = {
    "VN": 30, "US": 20, "GB": 10, "JP": 8, "KR": 8,
    "TH": 6,  "ID": 6,  "MY": 4,  "SG": 3, "PH": 3,
    "AU": 3,  "FR": 3,  "DE": 3,  "BR": 3, "MX": 2,
    "IN": 2,  "CA": 2,  "IT": 2,  "ES": 2, "NL": 2,
}

months = pd.date_range(start=START_MONTH, periods=N_MONTHS, freq="MS")
streaming_platform_ids = ["PLT_SPF", "PLT_APM", "PLT_TTK", "PLT_SCL", "PLT_YTM"]

# Only active tracks generate revenue
active_tracks = tracks_df[tracks_df["active_status"] == "ACTIVE"]

revenue_rows = []
for _, track in active_tracks.iterrows():
    # Each track active on 1-3 platforms
    track_platforms = random.sample(streaming_platform_ids, random.randint(1, 3))
    for platform_id in track_platforms:
        rate = platform_rates[platform_id]
        for month in months:
            month_key = month.strftime("%Y%m")
            # Sample 2-5 countries per track per platform per month
            n_countries = random.randint(2, 5)
            sampled_countries = random.choices(
                list(country_weights.keys()),
                weights=list(country_weights.values()),
                k=n_countries
            )
            for country in set(sampled_countries):
                # Realistic stream counts with seasonality
                base_streams = random.randint(500, 50000)
                seasonality  = 1.0 + 0.3 * np.sin(2 * np.pi * month.month / 12)
                streams      = int(base_streams * seasonality)
                revenue      = round(streams * rate, 4)
                revenue_rows.append({
                    "report_month"     : month_key,
                    "distributor_id"   : track["distributor_id"],
                    "platform_id"      : platform_id,
                    "artist_id"        : track["artist_id"],
                    "track_id"         : track["track_id"],
                    "country"          : country,
                    "streams"          : streams,
                    "revenue_original" : revenue,
                    "currency"         : "USD",
                    "exchange_rate"    : 1.0
                })

dist_revenue_df = pd.DataFrame(revenue_rows)
upload_df_to_bronze(dist_revenue_df, "distributor_revenue_monthly/", "distributor_revenue_monthly.csv")
print(f"\nDistributor revenue: {len(dist_revenue_df):,} rows")
print(f"Total streams : {dist_revenue_df['streams'].sum():,}")
print(f"Total revenue : ${dist_revenue_df['revenue_original'].sum():,.2f}")

Uploaded → bronze/distributor_revenue_monthly/distributor_revenue_monthly.csv (367,429 rows)

Distributor revenue: 367,429 rows
Total streams : 9,285,489,440
Total revenue : $23,694,934.05


In [19]:
# ── 3b. YouTube View Daily ────────────────────────────────────────────
date_range = pd.date_range(
    start=pd.Timestamp(START_MONTH),
    periods=N_MONTHS * 30,
    freq="D"
)

yt_view_rows = []
for _, video in videos_df.iterrows():
    # Each video active in 3-8 countries
    video_countries = random.sample([c[0] for c in COUNTRIES], random.randint(3, 8))
    upload_dt = pd.Timestamp(video["upload_date"])
    for date in date_range:
        if date < upload_dt:
            continue
        for country in video_countries:
            # Not every country gets views every day
            if random.random() < 0.6:
                continue
            # Power law view distribution - some videos much more popular
            base_views = int(np.random.pareto(1.5) * 1000) + 10
            yt_view_rows.append({
                "view_date" : date.strftime("%Y-%m-%d"),
                "video_id"  : video["video_id"],
                "country"   : country,
                "views"     : min(base_views, 500000)  # cap outliers
            })

yt_views_df = pd.DataFrame(yt_view_rows)
upload_df_to_bronze(yt_views_df, "youtube_view_daily/", "youtube_view_daily.csv")
print(f"\nYouTube views: {len(yt_views_df):,} rows")
print(f"Total views  : {yt_views_df['views'].sum():,}")

Uploaded → bronze/youtube_view_daily/youtube_view_daily.csv (394,637 rows)

YouTube views: 394,637 rows
Total views  : 755,362,573


In [20]:
# ── 3c. Content Contributor Mapping ──────────────────────────────────
contributor_ids = contributors_df["contributor_id"].tolist()
mapping_rows    = []

# Map tracks to contributors
for _, track in tracks_df.iterrows():
    n_contributors = random.randint(1, 4)
    track_contributors = random.sample(contributor_ids, n_contributors)
    for ctb_id in track_contributors:
        role = contributors_df[
            contributors_df["contributor_id"] == ctb_id
        ]["default_role"].values[0]
        mapping_rows.append({
            "asset_id"          : track["asset_id"],
            "contributor_id"    : ctb_id,
            "role_in_asset"     : role,
            "allocation_method" : "EQUAL",
            "allocation_percent": round(1.0 / n_contributors, 4)
        })

# Map videos to contributors
for _, video in videos_df.iterrows():
    n_contributors = random.randint(2, 5)
    video_contributors = random.sample(contributor_ids, n_contributors)
    for ctb_id in video_contributors:
        role = contributors_df[
            contributors_df["contributor_id"] == ctb_id
        ]["default_role"].values[0]
        mapping_rows.append({
            "asset_id"          : video["asset_id"],
            "contributor_id"    : ctb_id,
            "role_in_asset"     : role,
            "allocation_method" : "EQUAL",
            "allocation_percent": round(1.0 / n_contributors, 4)
        })

mapping_df = pd.DataFrame(mapping_rows)
upload_df_to_bronze(mapping_df, "content_contributor_mapping/", "content_contributor_mapping.csv")
print(f"\nContributor mapping: {len(mapping_df):,} rows")
print(f"Avg contributors per asset: {len(mapping_df)/(len(tracks_df)+len(videos_df)):.1f}")

Uploaded → bronze/content_contributor_mapping/content_contributor_mapping.csv (8,415 rows)

Contributor mapping: 8,415 rows
Avg contributors per asset: 2.5


## Step 4: Final Bronze Inventory

In [21]:
# List everything in Bronze
paginator = s3.get_paginator("list_objects_v2")
pages     = paginator.paginate(Bucket=BRONZE_BUCKET)

prefix_stats = {}
for page in pages:
    for obj in page.get("Contents", []):
        prefix = obj["Key"].split("/")[0]
        if prefix not in prefix_stats:
            prefix_stats[prefix] = {"count": 0, "size_mb": 0}
        prefix_stats[prefix]["count"]   += 1
        prefix_stats[prefix]["size_mb"] += obj["Size"] / 1024 / 1024

print("Bronze bucket inventory:")
print(f"{'Prefix':<35} {'Files':>6} {'Size':>10}")
print("-" * 55)
total_size = 0
for prefix, stats in sorted(prefix_stats.items()):
    size_str = f"{stats['size_mb']:.1f} MB" if stats['size_mb'] < 1024 else f"{stats['size_mb']/1024:.2f} GB"
    print(f"{prefix:<35} {stats['count']:>6} {size_str:>10}")
    total_size += stats['size_mb']
print("-" * 55)
print(f"{'TOTAL':<35} {'':>6} {total_size/1024:.2f} GB")

Bronze bucket inventory:
Prefix                               Files       Size
-------------------------------------------------------
artists                                  1     0.1 MB
content_contributor_mapping              1     0.3 MB
contributors                             1     0.0 MB
distributor_revenue_monthly              1    42.3 MB
distributors                             1     0.0 MB
listenbrainz                           100   18.74 GB
platforms                                1     0.0 MB
tracks                                   1     0.7 MB
youtube_channels                         1     0.0 MB
youtube_rate                             1     0.0 MB
youtube_videos                           1     0.0 MB
youtube_view_daily                       1     9.9 MB
-------------------------------------------------------
TOTAL                                      18.79 GB


In [22]:
# In your notebook
import io

def read_bronze_csv(prefix, filename):
    response = s3.get_object(Bucket=BRONZE_BUCKET, Key=f"{prefix}{filename}")
    return pd.read_csv(io.BytesIO(response["Body"].read()))

artists_check = read_bronze_csv("artists/", "artists.csv")
contributors_check = read_bronze_csv("contributors/", "contributors.csv")

print(f"Artists: {len(artists_check)} rows")
print(artists_check[["artist_id", "artist_name", "distributor_id"]].head(5).to_string())
print(f"\nContributors: {len(contributors_check)} rows")
print(contributors_check.head(5).to_string())

Artists: 500 rows
                              artist_id                       artist_name distributor_id
0  63ecc430-f223-4dbd-9e59-334284f11ad5                      SM Jazz Trio        DST_DTO
1  df8e73d9-8a1e-4312-9a4d-09528ab1062c                     Nicky Flowers        DST_RTN
2  d5fcca60-4499-432c-9482-e343813835f7                     Deine Freunde        DST_WMG
3  6007f29c-ab70-4a61-bace-b48714cb955f  Philly Joe Jones Big Band Sounds        DST_OPM
4  db1d5043-fa1b-4cb2-b566-8bf0480ab664                     Thomas Belhom        DST_TNC

Contributors: 50 rows
  contributor_id contributor_name          team     default_role active_status
0         CTB001    Nguyễn Văn An       Content   Artist Manager        ACTIVE
1         CTB002    Trần Thị Bình       Content         Arranger        ACTIVE
2         CTB003      Lê Văn Công  Distribution         Arranger        ACTIVE
3         CTB004    Phạm Thị Dung     Marketing         Arranger        ACTIVE
4         CTB005     Hoàng Văn

In [23]:
# ── END OF NOTEBOOK - RELEASE ALL MEMORY ─────────────────────────────
import gc
import ctypes

# All data is safely in Bronze MinIO
# No need to keep anything in memory

del recording_pool
del artist_pool
del sampled_recording_mbids
del sampled_artist_mbids
del artists_df
del tracks_df
del contributors_df
del channels_df
del videos_df
del dist_revenue_df
del yt_views_df
del mapping_df
del platforms
del distributors
del youtube_rate

# Force GC
gc.collect()

# Release to OS
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except:
    pass

print("All data safely in Bronze MinIO ✅")
print("Memory released ✅")
print("Kernel can now be safely restarted ✅")

All data safely in Bronze MinIO ✅
Memory released ✅
Kernel can now be safely restarted ✅
